# Tutorial 02 - Loading Data from Unstructured Directory

In Tutorial 01 we assumed a specific folder structure to load the audio files and create a PyTorch Dataset. This is restrictive as in most cases the dataset comes in a folder containing all audio files and the individual splits are determined by some other structure (e.g., `csv` or `json` files, etc.). In this Tutorial we demonstrate an alternative and more Pythonic-way to load your data and create the Audio Classification Dataset.

## 1. Dataset Downloading & Inspection

For the purposes of this Tutorial we use the SpeechCommands dataset, we use a small version of the dataset consisting of 12 spoken english commands (e.g., "down", "go", "left", etc.) from various speakers. More information about the dataset can be found in the [HEAR](https://arxiv.org/abs/2203.03022) evaluation benchmark dataset. 

In [1]:
# We download the dataset from zenodo using wget

!wget https://zenodo.org/records/5887964/files/hear2021-speech_commands-v0.0.2-5h-48000.tar.gz?download=1

--2026-04-17 15:24:25--  https://zenodo.org/records/5887964/files/hear2021-speech_commands-v0.0.2-5h-48000.tar.gz?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.185.48.75, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1430299345 (1.3G) [application/octet-stream]
Saving to: ‘hear2021-speech_commands-v0.0.2-5h-48000.tar.gz?download=1’

hear2021-speech_com 100%[===================>]   1.33G  6.69MB/s    in 4m 15s  

2026-04-17 15:28:40 (5.35 MB/s) - ‘hear2021-speech_commands-v0.0.2-5h-48000.tar.gz?download=1’ saved [1430299345/1430299345]



In [ ]:
# We extract the downloaded tar.gz file and move the contents to the /data directory (folder should exist)
!tar -zxf ./hear2021-speech_commands-v0.0.2-5h-48000.tar.gz?download=1 -C /data

Now the dataset is available at `/data/hear-2021.0.6/tasks/speech_commands-v0.0.2-5h`. The folder contains the following files:

- labelvocabulary.csv: Containing the class mapping between class names and integer values.
- task_metadata.json: Metadata of the dataset
- train.json: The audio filenames corresponding to the training set.
- valid.json: The audio filenames corresponding to the validation set.
- test.json: The audio filenames corresponding to the test set.

The folder `48000` contains three subfolders `train`, `test`, `valid`, each containing the respective audio files of the specified split in 48KHz sampling rate format.

In [5]:
# We inspect the contests of the medatata file
import json
from pathlib import Path

DATA_PATH = Path("/data/hear-2021.0.6/tasks/speech_commands-v0.0.2-5h/")
TRAIN_PATH = DATA_PATH / "48000" / "train"
TEST_PATH = DATA_PATH / "48000" / "test"
VALID_PATH = DATA_PATH / "48000" / "valid"

with open(DATA_PATH / "task_metadata.json", "r") as f:
    metadata = json.load(f)

metadata

{'task_name': 'speech_commands',
 'version': 'v0.0.2',
 'embedding_type': 'scene',
 'prediction_type': 'multiclass',
 'split_mode': 'trainvaltest',
 'sample_duration': 1.0,
 'evaluation': ['top1_acc'],
 'download_urls': [{'split': 'train',
   'url': 'http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz',
   'md5': '6b74f3901214cb2c2934e98196829835'},
  {'split': 'test',
   'url': 'http://download.tensorflow.org/data/speech_commands_test_set_v0.02.tar.gz',
   'md5': '854c580ee90bff80c516491c84544e32'}],
 'default_mode': '5h',
 'max_task_duration_by_split': {'train': 16000.0,
  'valid': 2000.0,
  'test': None},
 'tmp_dir': '_workdir',
 'mode': '5h',
 'splits': ['train', 'valid', 'test']}

Through the metadata we see that each audio is 1-second long. Therefore, we will set `segment_duration=1.0` for creating the PyTorch dataset. Below we inspect the format of the json splitting files.

In [7]:
with open(DATA_PATH / "train.json", "r") as f:
    train_json = json.load(f)
    
# Inspect the first entry in the train.json file
key, value = next(iter(train_json.items()))

print(key, value)

_silence__doing_the_dishes-1048000.wav ['_silence_']


We see that the json maps the filenames to the individual classes. We parse the json files for the validation / test splits in similar manner.

In [8]:
with open(DATA_PATH / "test.json", "r") as f:
    test_json = json.load(f)
    
with open(DATA_PATH / "valid.json", "r") as f:
    valid_json = json.load(f)

## 2. Dataset Creation using Python Dictionaries


Now that we understand the structure of the dataset we can easily create the datasets. We first define the `class_mapping` through the `labelvocabulary.csv` file which is available.

In [12]:
import csv

with open(DATA_PATH / "labelvocabulary.csv", "r") as f:
    reader = csv.reader(f)
    next(reader)  # Skip the header row
    label_mapping = {rows[0]: rows[1] for rows in reader}
    
class_mapping = {v: int(k) for k, v in label_mapping.items()}

class_mapping

{'_silence_': 0,
 '_unknown_': 1,
 'down': 2,
 'go': 3,
 'left': 4,
 'no': 5,
 'off': 6,
 'on': 7,
 'right': 8,
 'stop': 9,
 'up': 10,
 'yes': 11}

To instantiate a PyTorch Dataset for audio classification we use the method `audio_classification_dataset_from_dictionary`. The method expects the same arguments as the `audio_classification_dataset_from_dir` with the exception that instead of a path we provide a Python dictionary of the form `{"<abs_path_to_file>": "class_name"}`. This is handled by the `file_to_class_mapping` argument. Luckily for us, this information is contained in the `train_json, valid_json`, and `test_json` variables defined previously.

In [15]:
from deepaudiox import audio_classification_dataset_from_dictionary

# We only need to prepend the absolute path and index the class label for the dataset
train_json = {str(TRAIN_PATH / key): value[0] for key, value in train_json.items()}
valid_json = {str(VALID_PATH / key): value[0] for key, value in valid_json.items()}
test_json = {str(TEST_PATH / key): value[0] for key, value in test_json.items()}

train_dset = audio_classification_dataset_from_dictionary(file_to_class_mapping=train_json,
                                                          class_mapping=class_mapping,
                                                          sample_rate=32000,
                                                          segment_duration=1.0)

valid_dset = audio_classification_dataset_from_dictionary(file_to_class_mapping=valid_json,
                                                          class_mapping=class_mapping,
                                                          sample_rate=32000,
                                                          segment_duration=1.0)

test_dset = audio_classification_dataset_from_dictionary(file_to_class_mapping=test_json,
                                                          class_mapping=class_mapping,
                                                          sample_rate=32000,
                                                          segment_duration=1.0)

In [17]:
# Check the first entry in the training dataset
print(train_dset[0])

{'path': '/data/hear-2021.0.6/tasks/speech_commands-v0.0.2-5h/48000/train/_silence__doing_the_dishes-1048000.wav', 'y_true': 0, 'class_name': '_silence_', 'segment_idx': 0, 'feature': array([ 0.01144081,  0.00943983,  0.00135719, ..., -0.01853629,
       -0.0183027 , -0.0120908 ], shape=(32000,), dtype=float32)}


In [18]:
# Check the lengths of the datasets
print(f"Number of training samples: {len(train_dset)}")
print(f"Number of validation samples: {len(valid_dset)}")
print(f"Number of test samples: {len(test_dset)}")

Number of training samples: 16000
Number of validation samples: 2000
Number of test samples: 4890


## 3. Initializing the AudioClassifier

Now the rest is easy. The steps are Classifier Initialization -> Trainer -> Evaluator. We instantiate a simple audio classifier using MobileNet as backbone feature extractor - a lightweight CNN-based architecture enabling fast training. Since the backbone is lightweight we train it from scratch.

In [20]:
from deepaudiox import AudioClassifier

model = AudioClassifier(backbone="mobilenet_10_as",
                        num_classes=len(class_mapping),
                        freeze_backbone=False,
                        pretrained=True,
                        sample_rate=32_000)

To see all the available backbones on the library use the `AVAILABLE_BACKBONES` variable lists all backbones.

In [21]:
from deepaudiox import AVAILABLE_BACKBONES

In [24]:
# Model Inspection
model

AudioClassifierConstructor(
  (backbone_constructor): BackboneConstructor(
    (backbone): MobileNet(
      (features): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (1): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            )
          )
 

## 4. Training

Now we are ready to train our model for speech command classification. Note that in this case, the dataset comes with a predetermined validation dataset where we can utilize during training.

In [25]:
from deepaudiox import Trainer

trainer = Trainer(model=model,
                  train_dset=train_dset,
                  validation_dset=valid_dset,
                  epochs=50,
                  batch_size=128,
                  patience=10)

trainer.train()

[Epoch 1/50]


Using GPU: NVIDIA GeForce RTX 4090


Epoch 1 | Train Loss: 1.5644 | Val. Loss: 1.5606 | Time: 3.32s      
[CHECKPOINTER] Validation loss decreased: (inf --> 1.560594), (-nan%).
[CHECKPOINTER] Checkpoint saved successfully at: checkpoint.pt
[Epoch 2/50]
Epoch 2 | Train Loss: 1.4823 | Val. Loss: 0.9147 | Time: 2.62s      
[CHECKPOINTER] Validation loss decreased: (1.560594 --> 0.914667), (-41.39%).
[CHECKPOINTER] Checkpoint saved successfully at: checkpoint.pt
[Epoch 3/50]
Epoch 3 | Train Loss: 1.3784 | Val. Loss: 1.4767 | Time: 2.66s      
[Epoch 4/50]
Epoch 4 | Train Loss: 1.3140 | Val. Loss: 0.4436 | Time: 2.60s      
[CHECKPOINTER] Validation loss decreased: (0.914667 --> 0.443601), (-51.50%).
[CHECKPOINTER] Checkpoint saved successfully at: checkpoint.pt
[Epoch 5/50]
Epoch 5 | Train Loss: 1.3024 | Val. Loss: 0.3455 | Time: 2.65s      
[CHECKPOINTER] Validation loss decreased: (0.443601 --> 0.345517), (-22.11%).
[CHECKPOINTER] Checkpoint saved successfully at: checkpoint.pt
[Epoch 6/50]
Epoch 6 | Train Loss: 1.2830 | Va

## 5. Evaluation

In similar manner as in the first tutorial, we use the `Evaluator` to check the performance on the held-out test set.

In [26]:
from deepaudiox import Evaluator

# First load the best model checkpoint
model = AudioClassifier.from_checkpoint("checkpoint.pt")

evaluator = Evaluator(model=model, test_dset=test_dset, class_mapping=class_mapping)

evaluator.evaluate() 

Using GPU: NVIDIA GeForce RTX 4090


Testing has finished.                                                  
[REPORTER] Class mapping: {'_silence_': 0, '_unknown_': 1, 'down': 2, 'go': 3, 'left': 4, 'no': 5, 'off': 6, 'on': 7, 'right': 8, 'stop': 9, 'up': 10, 'yes': 11} 

[REPORTER] Classification Report: 

              precision    recall  f1-score   support

   _silence_       1.00      0.94      0.97       408
   _unknown_       0.60      0.99      0.75       408
        down       0.96      0.85      0.90       406
          go       0.98      0.80      0.88       402
        left       0.98      0.92      0.95       412
          no       0.91      0.93      0.92       405
         off       0.97      0.92      0.95       402
          on       0.99      0.87      0.93       396
       right       1.00      0.93      0.96       396
        stop       1.00      0.99      0.99       411
          up       0.97      0.96      0.97       425
         yes       0.99      0.98      0.99       419

    accuracy            